# Stage 1: Age-18 Activity Outcome Construction

# Part 1: Source Review

In [1]:
# 1: Set project paths and import packages

from pathlib import Path

import pandas as pd
from IPython.display import display

working_directory = Path.cwd().resolve()

project_root = next(
    (
        path
        for path in [working_directory, *working_directory.parents]
        if (path / "data_working").exists()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Project root could not be located. Expected a data_working directory "
        "in the current directory or one of its parents."
    )

source_directory = project_root / "data_working"

output_directory = (
    project_root
    / "data_derived"
    / "stage_1_outcome_construction"
)

print("Project root located.")
print("Source directory:", source_directory.relative_to(project_root))
print("Output directory:", output_directory.relative_to(project_root))


Project root located.
Source directory: data_working
Output directory: data_derived\stage_1_outcome_construction


In [2]:
# 2: Register the available Stata files

stata_files = sorted(
    source_directory.rglob("*.dta")
)

if not stata_files:
    raise FileNotFoundError(
        "No Stata files were found under data_working."
    )

stata_file_register = pd.DataFrame(
    {
        "source_file": [
            file_path.name
            for file_path in stata_files
        ],
        "relative_path": [
            str(file_path.relative_to(project_root))
            for file_path in stata_files
        ],
        "size_mb": [
            round(file_path.stat().st_size / (1024 ** 2), 2)
            for file_path in stata_files
        ],
    }
)

print("Stata files found:", len(stata_file_register))

Stata files found: 53


In [3]:
# 3: Search variable labels for the age-18 reference point

search_terms = (
    "may 2009",
    "main activity at 18",
)

source_review_records = []
unreadable_files = []

for file_path in stata_files:
    try:
        with pd.io.stata.StataReader(file_path) as reader:
            file_variable_labels = reader.variable_labels()

        for variable, label in file_variable_labels.items():
            label_text = str(label)
            label_lower = label_text.lower()

            if any(
                term in label_lower
                for term in search_terms
            ):
                source_review_records.append(
                    {
                        "source_file": file_path.name,
                        "source_path": str(
                            file_path.relative_to(project_root)
                        ),
                        "variable": variable,
                        "variable_label": label_text,
                    }
                )

    except Exception as error:
        unreadable_files.append(
            {
                "source_file": file_path.name,
                "source_path": str(
                    file_path.relative_to(project_root)
                ),
                "error": str(error),
            }
        )

source_review = pd.DataFrame(source_review_records)

if source_review.empty:
    raise ValueError(
        "No variables matched the source-review terms."
    )

source_review = (
    source_review
    .drop_duplicates()
    .sort_values(
        [
            "source_file",
            "variable",
        ]
    )
    .reset_index(drop=True)
)

display(source_review)

if unreadable_files:
    print()
    print("Files not read during the metadata review:")
    display(pd.DataFrame(unreadable_files))

,source_file,source_path,variable,variable_label
0,lsype_main_activity_w4-7_nov2011_suppressed.dta,data_working\nextsteps_5545\UKDA-5545-stata\st...,W7FinAct53_B11,DV: Main Activity May 2009 - Bulletin 11 (Main...


In [4]:
# 4: Select the May 2009 main-activity endpoint

endpoint_candidates = source_review.loc[
    source_review["variable_label"].str.contains(
        "may 2009",
        case=False,
        na=False,
    )
    &
    source_review["variable_label"].str.contains(
        "main activity at 18",
        case=False,
        na=False,
    )
].copy()

print("Endpoint candidates:", len(endpoint_candidates))

if len(endpoint_candidates) != 1:
    raise ValueError(
        "The source review did not identify one unambiguous "
        "May 2009 main-activity endpoint."
    )

selected_endpoint = endpoint_candidates.iloc[0]

monthly_activity_path = (
    project_root
    / selected_endpoint["source_path"]
)

endpoint_variable = selected_endpoint["variable"]
endpoint_variable_label = selected_endpoint["variable_label"]

if not monthly_activity_path.exists():
    raise FileNotFoundError(
        f"Selected source file not found: {monthly_activity_path}"
    )

print("Selected source file:")
print(f"- {monthly_activity_path.name}")

print()
print("Selected endpoint variable:")
print(f"- {endpoint_variable}: {endpoint_variable_label}")

Endpoint candidates: 1
Selected source file:
- lsype_main_activity_w4-7_nov2011_suppressed.dta

Selected endpoint variable:
- W7FinAct53_B11: DV: Main Activity May 2009 - Bulletin 11 (Main Activity at 18)


# Part 2: Endpoint Inspection and Outcome Definition

In [5]:
# 1: Read the selected endpoint as codes and source labels

source_raw = pd.read_stata(
    monthly_activity_path,
    columns=[
        "NSID",
        endpoint_variable,
    ],
    convert_categoricals=False,
)

source_labelled = pd.read_stata(
    monthly_activity_path,
    columns=[
        "NSID",
        endpoint_variable,
    ],
    convert_categoricals=True,
)

if len(source_raw) != len(source_labelled):
    raise ValueError(
        "The coded and labelled source reads returned "
        "different numbers of rows."
    )

raw_ids = source_raw["NSID"].astype("string")
labelled_ids = source_labelled["NSID"].astype("string")

if not raw_ids.equals(labelled_ids):
    raise ValueError(
        "Participant order changed between the coded "
        "and labelled source reads."
    )

endpoint_review = pd.DataFrame(
    {
        "NSID": raw_ids,
        "raw_code": source_raw[endpoint_variable],
        "source_label": source_labelled[
            endpoint_variable
        ].astype("string"),
    }
)

print("Rows:", len(endpoint_review))
print(
    "Unique NSID:",
    endpoint_review["NSID"].nunique(dropna=True),
)
print(
    "Duplicate NSID:",
    endpoint_review["NSID"].duplicated().sum(),
)
print(
    "Missing NSID:",
    endpoint_review["NSID"].isna().sum(),
)
print(
    "Missing endpoint codes:",
    endpoint_review["raw_code"].isna().sum(),
)

Rows: 11811
Unique NSID: 11811
Duplicate NSID: 0
Missing NSID: 0
Missing endpoint codes: 0


In [6]:
# 2: Inspect the observed source codes and labels

code_label_pairs = (
    endpoint_review.loc[
        endpoint_review["raw_code"].notna()
        &
        endpoint_review["source_label"].notna(),
        [
            "raw_code",
            "source_label",
        ],
    ]
    .drop_duplicates()
    .sort_values("raw_code")
    .reset_index(drop=True)
)

labels_per_code = (
    code_label_pairs
    .groupby("raw_code")["source_label"]
    .nunique()
)

codes_per_label = (
    code_label_pairs
    .groupby("source_label")["raw_code"]
    .nunique()
)

if (labels_per_code > 1).any():
    raise ValueError(
        "At least one source code had more than one label."
    )

if (codes_per_label > 1).any():
    raise ValueError(
        "At least one source label had more than one code."
    )

source_distribution = (
    endpoint_review
    .groupby(
        [
            "raw_code",
            "source_label",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        "raw_code",
        na_position="last",
    )
    .reset_index(drop=True)
)

source_distribution["percentage_of_source"] = (
    source_distribution["count"]
    / len(endpoint_review)
    * 100
)

display(code_label_pairs)
display(source_distribution)

if endpoint_review["NSID"].isna().any():
    raise ValueError(
        "Missing participant identifiers were found."
    )

if not endpoint_review["NSID"].is_unique:
    raise ValueError(
        "Duplicate participant identifiers were found."
    )

,raw_code,source_label
0,-94,Insufficient information
1,1,Education
2,4,Employed
3,5,Apprenticeship/training
4,6,Unemployed/Inactive (NEET)


,raw_code,source_label,count,percentage_of_source
0,-94,Insufficient information,2044,17.305901
1,1,Education,5132,43.451020
2,4,Employed,2831,23.969181
3,5,Apprenticeship/training,524,4.436542
4,6,Unemployed/Inactive (NEET),1280,10.837355


## Endpoint Record Coverage

The monthly activity source file contained 11,811 participants, with one record per participant and no missing endpoint codes.

Of these records, 2,044 participants (17.31%) were coded as having insufficient information. The remaining 9,767 participants (82.69%) had one of the four substantive May 2009 activity codes and were eligible for inclusion in the primary outcome dataset.

The observed source categories were:

- Education: 5,132 participants
- Employed: 2,831 participants
- Apprenticeship/training: 524 participants
- Unemployed/Inactive (NEET): 1,280 participants
- Insufficient information: 2,044 participants

In [7]:
# 3: Assign the analytical outcome labels

analytical_label_by_source_label = {
    "Education": "Education",
    "Employed": "Employment",
    "Apprenticeship/training": (
        "Apprenticeship or training"
    ),
    "Unemployed/Inactive (NEET)": (
        "Unemployment or inactivity (NEET)"
    ),
}

excluded_source_labels = {
    "Insufficient information",
}

observed_source_labels = set(
    endpoint_review["source_label"]
    .dropna()
    .unique()
)

unassigned_source_labels = (
    observed_source_labels
    - set(analytical_label_by_source_label)
    - excluded_source_labels
)

if unassigned_source_labels:
    raise ValueError(
        "Observed source labels were not assigned to "
        "the analytical outcome or the excluded group: "
        f"{sorted(unassigned_source_labels)}"
    )

label_definition = code_label_pairs.copy()

label_definition["analysis_label"] = (
    label_definition["source_label"]
    .map(analytical_label_by_source_label)
)

label_definition["included_in_outcome"] = (
    label_definition["analysis_label"].notna()
)

display(label_definition)

,raw_code,source_label,analysis_label,included_in_outcome
0,-94,Insufficient information,NaN,False
1,1,Education,Education,True
2,4,Employed,Employment,True
3,5,Apprenticeship/training,Apprenticeship or training,True
4,6,Unemployed/Inactive (NEET),Unemployment or inactivity (NEET),True


## Outcome Definition

The four substantive source categories were assigned noun-form analytical labels:

- Education
- Employment
- Apprenticeship or training
- Unemployment or inactivity (NEET)

Records labelled as insufficient information were excluded from the participant-level outcome dataset.

# Part 3: Primary Outcome Construction

In [8]:
# 1: Construct the participant-level outcome

outcome_source = endpoint_review.loc[
    endpoint_review["source_label"].isin(
        analytical_label_by_source_label
    )
].copy()

age18_outcome = pd.DataFrame(
    {
        "NSID": outcome_source["NSID"],
        "age18_outcome_code": outcome_source[
            "raw_code"
        ].astype("int64"),
        "age18_outcome": outcome_source[
            "source_label"
        ].map(
            analytical_label_by_source_label
        ),
    }
)

outcome_categories = list(
    analytical_label_by_source_label.values()
)

age18_outcome["age18_outcome"] = pd.Categorical(
    age18_outcome["age18_outcome"],
    categories=outcome_categories,
    ordered=False,
)

age18_outcome = (
    age18_outcome
    .sort_values("NSID")
    .reset_index(drop=True)
)

print("Outcome rows:", len(age18_outcome))
print(
    "Unique NSID:",
    age18_outcome["NSID"].nunique(),
)
print(
    "Duplicate NSID:",
    age18_outcome["NSID"].duplicated().sum(),
)
print(
    "Missing values:",
    age18_outcome.isna().sum().sum(),
)

Outcome rows: 9767
Unique NSID: 9767
Duplicate NSID: 0
Missing values: 0


In [9]:
# 2: Check sample flow and outcome distribution

source_n = len(endpoint_review)

insufficient_information_n = (
    endpoint_review["source_label"]
    .isin(excluded_source_labels)
    .sum()
)

missing_endpoint_n = (
    endpoint_review["raw_code"].isna()
    |
    endpoint_review["source_label"].isna()
).sum()

valid_outcome_n = len(age18_outcome)

if source_n != (
    insufficient_information_n
    + missing_endpoint_n
    + valid_outcome_n
):
    raise ValueError(
        "The source records were not fully accounted for."
    )

if age18_outcome["NSID"].isna().any():
    raise ValueError(
        "Missing participant identifiers were found "
        "in the outcome dataset."
    )

if not age18_outcome["NSID"].is_unique:
    raise ValueError(
        "Duplicate participant identifiers were found "
        "in the outcome dataset."
    )

if age18_outcome[
    [
        "age18_outcome_code",
        "age18_outcome",
    ]
].isna().any().any():
    raise ValueError(
        "Missing values were found in the outcome dataset."
    )

outcome_distribution = (
    age18_outcome["age18_outcome"]
    .value_counts(sort=False)
    .rename_axis("age18_outcome")
    .reset_index(name="count")
)

outcome_distribution["percentage"] = (
    outcome_distribution["count"]
    / valid_outcome_n
    * 100
)

sample_flow = pd.DataFrame(
    {
        "stage": [
            "Monthly activity source file",
            "Insufficient information",
            "Missing endpoint value",
            "Valid primary outcome",
        ],
        "count": [
            source_n,
            insufficient_information_n,
            missing_endpoint_n,
            valid_outcome_n,
        ],
        "percentage_of_source": [
            100.0,
            insufficient_information_n
            / source_n
            * 100,
            missing_endpoint_n
            / source_n
            * 100,
            valid_outcome_n
            / source_n
            * 100,
        ],
    }
)

display(sample_flow)
display(outcome_distribution)

,stage,count,percentage_of_source
0,Monthly activity source file,11811,100.000000
1,Insufficient information,2044,17.305901
2,Missing endpoint value,0,0.000000
3,Valid primary outcome,9767,82.694099


,age18_outcome,count,percentage
0,Education,5132,52.544282
1,Employment,2831,28.985359
2,Apprenticeship or training,524,5.365005
3,Unemployment or inactivity (NEET),1280,13.105355


In [10]:
# 3: Validate the insufficient-information exclusion

insufficient_code = -94
insufficient_label = "Insufficient information"

insufficient_records = endpoint_review.loc[
    endpoint_review["raw_code"].eq(insufficient_code),
    [
        "NSID",
        "raw_code",
        "source_label",
    ],
].copy()

observed_labels = set(
    insufficient_records["source_label"]
    .dropna()
    .astype(str)
    .unique()
)

if observed_labels != {insufficient_label}:
    raise ValueError(
        "Code -94 was not consistently labelled as "
        "'Insufficient information'."
    )

codes_for_insufficient_label = set(
    endpoint_review.loc[
        endpoint_review["source_label"].eq(insufficient_label),
        "raw_code",
    ]
    .dropna()
    .unique()
)

if codes_for_insufficient_label != {insufficient_code}:
    raise ValueError(
        "The 'Insufficient information' label was associated "
        "with an unexpected source code."
    )

source_ids = set(endpoint_review["NSID"])
outcome_ids = set(age18_outcome["NSID"])

excluded_ids = source_ids - outcome_ids
insufficient_ids = set(insufficient_records["NSID"])

if excluded_ids != insufficient_ids:
    raise ValueError(
        "The participants excluded from the outcome dataset "
        "did not correspond exactly to code -94."
    )

insufficient_n = len(insufficient_records)
source_n = len(endpoint_review)
valid_outcome_n = len(age18_outcome)

exclusion_check = pd.DataFrame(
    {
        "check": [
            "Source code",
            "Source label",
            "Participants with insufficient information",
            "Percentage of source records",
            "Participants included in the outcome dataset",
            "Other participants excluded",
        ],
        "result": [
            insufficient_code,
            insufficient_label,
            insufficient_n,
            round(insufficient_n / source_n * 100, 2),
            valid_outcome_n,
            len(excluded_ids - insufficient_ids),
        ],
    }
)

display(exclusion_check)

,check,result
0,Source code,-94
1,Source label,Insufficient information
2,Participants with insufficient information,2044
3,Percentage of source records,17.31
4,Participants included in the outcome dataset,9767
5,Other participants excluded,0


## Insufficient-Information Records

Source code `-94` was consistently labelled as `Insufficient information`. It applied to 2,044 participants, representing 17.31% of the monthly activity source file.

These records corresponded exactly to the participants not included in the primary outcome dataset. No participants with one of the four substantive May 2009 activity categories were excluded.

This check confirms that outcome exclusion followed the source-defined endpoint coding. It does not explain why individual participants received the insufficient-information code; outcome data availability will therefore be examined separately in the outcome EDA.

## Sample Flow and Outcome Distribution

The monthly activity source file contained 11,811 participants. After excluding the 2,044 records coded as having insufficient information, the primary outcome dataset contained 9,767 participants with one of the four substantive May 2009 activity outcomes.

Education was the largest category (52.54%), followed by Employment (28.99%), Unemployment or inactivity (13.11%), and Apprenticeship or training (5.37%). The outcome distribution was imbalanced, with Education accounting for more than half of the analytical sample.

# Part 4: Output Files

The participant-level outcome dataset and supporting construction tables are saved for use in subsequent analysis. The saved files are then reloaded and checked for row counts, unique participant identifiers, missing values and outcome categories.


In [11]:
# 1: Prepare the output tables

output_directory.mkdir(
    parents=True,
    exist_ok=True,
)

file_register_path = (
    output_directory
    / "stage_1_stata_file_register.csv"
)

source_review_path = (
    output_directory
    / "stage_1_source_review.csv"
)

outcome_path = (
    output_directory
    / "age18_activity_outcome.csv"
)

sample_flow_path = (
    output_directory
    / "age18_activity_outcome_sample_flow.csv"
)

distribution_path = (
    output_directory
    / "age18_activity_outcome_distribution.csv"
)

codebook_path = (
    output_directory
    / "age18_activity_outcome_codebook.csv"
)

outcome_export = age18_outcome.copy()

outcome_export["age18_outcome"] = (
    outcome_export["age18_outcome"]
    .astype("string")
)

observed_counts = (
    endpoint_review["raw_code"]
    .value_counts()
    .to_dict()
)

outcome_codebook = label_definition.copy()

outcome_codebook.insert(
    0,
    "source_dataset",
    monthly_activity_path.name,
)

outcome_codebook.insert(
    1,
    "source_variable",
    endpoint_variable,
)

outcome_codebook.insert(
    2,
    "source_variable_label",
    endpoint_variable_label,
)

outcome_codebook.insert(
    3,
    "reference_period",
    "May 2009",
)

outcome_codebook["observed_count"] = (
    outcome_codebook["raw_code"]
    .map(observed_counts)
    .astype("int64")
)

output_paths = {
    "Stata file register": file_register_path,
    "Source review": source_review_path,
    "Outcome dataset": outcome_path,
    "Sample-flow table": sample_flow_path,
    "Outcome distribution": distribution_path,
    "Outcome codebook": codebook_path,
}

for output_name, path in output_paths.items():
    print(
        f"- {output_name}: "
        f"{path.relative_to(project_root)}"
    )

- Stata file register: data_derived\stage_1_outcome_construction\stage_1_stata_file_register.csv
- Source review: data_derived\stage_1_outcome_construction\stage_1_source_review.csv
- Outcome dataset: data_derived\stage_1_outcome_construction\age18_activity_outcome.csv
- Sample-flow table: data_derived\stage_1_outcome_construction\age18_activity_outcome_sample_flow.csv
- Outcome distribution: data_derived\stage_1_outcome_construction\age18_activity_outcome_distribution.csv
- Outcome codebook: data_derived\stage_1_outcome_construction\age18_activity_outcome_codebook.csv


In [12]:
# 2: Save the Stage 1 outputs

stata_file_register.to_csv(
    file_register_path,
    index=False,
)

source_review.to_csv(
    source_review_path,
    index=False,
)

outcome_export.to_csv(
    outcome_path,
    index=False,
)

sample_flow.to_csv(
    sample_flow_path,
    index=False,
)

outcome_distribution.to_csv(
    distribution_path,
    index=False,
)

outcome_codebook.to_csv(
    codebook_path,
    index=False,
)

print("Stage 1 output files saved.")

Stage 1 output files saved.


In [13]:
# 3: Reload and verify the saved files

file_register_reloaded = pd.read_csv(
    file_register_path
)

source_review_reloaded = pd.read_csv(
    source_review_path
)

outcome_reloaded = pd.read_csv(
    outcome_path,
    dtype={
        "NSID": "string",
        "age18_outcome_code": "int64",
        "age18_outcome": "string",
    },
)

sample_flow_reloaded = pd.read_csv(
    sample_flow_path
)

distribution_reloaded = pd.read_csv(
    distribution_path
)

codebook_reloaded = pd.read_csv(
    codebook_path,
    dtype={
        "included_in_outcome": "boolean",
    },
)

pd.testing.assert_frame_equal(
    file_register_reloaded,
    stata_file_register,
    check_dtype=False,
)

pd.testing.assert_frame_equal(
    source_review_reloaded,
    source_review,
    check_dtype=False,
)

expected_outcome = outcome_export.copy()

expected_outcome["NSID"] = (
    expected_outcome["NSID"]
    .astype("string")
)

expected_outcome["age18_outcome"] = (
    expected_outcome["age18_outcome"]
    .astype("string")
)

pd.testing.assert_frame_equal(
    outcome_reloaded,
    expected_outcome,
    check_dtype=False,
)

expected_sample_flow_counts = (
    sample_flow
    .set_index("stage")["count"]
    .to_dict()
)

observed_sample_flow_counts = (
    sample_flow_reloaded
    .set_index("stage")["count"]
    .to_dict()
)

if observed_sample_flow_counts != expected_sample_flow_counts:
    raise ValueError(
        "The saved sample-flow counts changed after reload."
    )

if source_n != (
    insufficient_information_n
    + missing_endpoint_n
    + valid_outcome_n
):
    raise ValueError(
        "The sample-flow totals were inconsistent."
    )

saved_distribution_counts = (
    distribution_reloaded
    .set_index("age18_outcome")["count"]
    .to_dict()
)

observed_distribution_counts = (
    outcome_reloaded["age18_outcome"]
    .value_counts()
    .to_dict()
)

if saved_distribution_counts != observed_distribution_counts:
    raise ValueError(
        "The saved distribution did not match "
        "the participant-level outcome."
    )

if distribution_reloaded["count"].sum() != valid_outcome_n:
    raise ValueError(
        "The saved distribution did not sum "
        "to the valid outcome sample."
    )

if abs(
    distribution_reloaded["percentage"].sum()
    - 100
) >= 0.01:
    raise ValueError(
        "The saved outcome percentages did not sum to 100."
    )

if (
    codebook_reloaded.loc[
        codebook_reloaded["included_in_outcome"],
        "observed_count",
    ].sum()
    != valid_outcome_n
):
    raise ValueError(
        "The included codebook counts did not match "
        "the valid outcome sample."
    )

for path in output_paths.values():
    if not path.exists():
        raise FileNotFoundError(
            f"Missing output file: {path}"
        )

print("Verified output files:")

for output_name, path in output_paths.items():
    print(f"- {output_name}: {path.name}")

Verified output files:
- Stata file register: stage_1_stata_file_register.csv
- Source review: stage_1_source_review.csv
- Outcome dataset: age18_activity_outcome.csv
- Sample-flow table: age18_activity_outcome_sample_flow.csv
- Outcome distribution: age18_activity_outcome_distribution.csv
- Outcome codebook: age18_activity_outcome_codebook.csv


# Part 5: Construction Summary

In [14]:
# 1: Summarise the construction results

print("Selected source file:")
print(f"- {monthly_activity_path.name}")

print()
print("Selected endpoint variable:")
print(f"- {endpoint_variable}: {endpoint_variable_label}")

print()
print("Sample flow:")
print(f"- Source participants: {source_n:,}")
print(
    "- Insufficient information: "
    f"{insufficient_information_n:,}"
)
print(
    "- Missing endpoint value: "
    f"{missing_endpoint_n:,}"
)
print(f"- Valid primary outcome: {valid_outcome_n:,}")

print()
print("Analytical outcome categories:")

for category in outcome_categories:
    print(f"- {category}")

print()
print("Saved outputs:")

for output_name, path in output_paths.items():
    print(f"- {output_name}: {path.name}")

Selected source file:
- lsype_main_activity_w4-7_nov2011_suppressed.dta

Selected endpoint variable:
- W7FinAct53_B11: DV: Main Activity May 2009 - Bulletin 11 (Main Activity at 18)

Sample flow:
- Source participants: 11,811
- Insufficient information: 2,044
- Missing endpoint value: 0
- Valid primary outcome: 9,767

Analytical outcome categories:
- Education
- Employment
- Apprenticeship or training
- Unemployment or inactivity (NEET)

Saved outputs:
- Stata file register: stage_1_stata_file_register.csv
- Source review: stage_1_source_review.csv
- Outcome dataset: age18_activity_outcome.csv
- Sample-flow table: age18_activity_outcome_sample_flow.csv
- Outcome distribution: age18_activity_outcome_distribution.csv
- Outcome codebook: age18_activity_outcome_codebook.csv


## Stage 1 Outcome Construction Summary

The May 2009 main-activity variable `W7FinAct53_B11` was selected as the primary age-18 outcome. Of 11,811 participants, 2,044 were coded as having insufficient information, leaving 9,767 participants with a valid outcome.

The four analytical categories were Education, Employment, Apprenticeship or training, and Unemployment or inactivity (NEET). The participant-level outcome dataset and supporting source-review, sample-flow, distribution and codebook files were saved for subsequent analysis.